In [1]:
cd ..

/home/karaman/Documents/bet


In [2]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.churn_label_generator import generate_churn_labels



In [28]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_bronze = spark.read.parquet("./data/bronze/players")
sessions_bronze = spark.read.parquet("./data/bronze/sessions")
transactions_bronze = spark.read.parquet("./data/bronze/transactions")


In [30]:
transactions_bronze.filter(F.col('success_flag')==False).count()


4073

In [31]:
transactions_bronze = (
    transactions_bronze
    .withColumn(
        "signed_amount", 
        F.when(F.col("transaction_type") == "deposit", F.col("amount") * F.col("success_flag").cast('int'))
         .when(F.col("transaction_type") == "withdrawal", -F.col("amount") * F.col("success_flag").cast('int'))
         .otherwise(F.lit(0.0))

    )
)